In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch
import psutil
import os

print("="*50)
print("       SYSTEM INFRASTRUCTURE REPORT")
print("="*50)

print("\n[Hardware Core - GPU]")
if torch.cuda.is_available():
    print(f"Entity: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Attributes: {vram:.2f} GB GDDR6 VRAM")
else:
    print("Entity: No GPU detected or CUDA not configured.")

print("\n[System Memory - RAM]")
ram = psutil.virtual_memory().total / (1024**3)
print(f"Entity: Random Access Memory (RAM)")
print(f"Attributes: {ram:.2f} GB System Allocation")

print("\n[Core Framework]")
print(f"Engine: PyTorch Framework Architecture")
print(f"Version: {torch.__version__}")

print("\n[Parallel Computing Platform]")
print(f"Ecosystem: NVIDIA CUDA Ecosystem")
print(f"Version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print("="*50)

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0))


In [ ]:
import os

class Config:
    #BASE_DIR = "/content/drive/MyDrive/Thesis_dataset"
    BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()

    DATASET_DIR = os.path.join(BASE_DIR, "dataset/chunks")
    SPLIT_DIR = os.path.join(BASE_DIR, "dataset/splits")

    MODEL_DIR = os.path.join(BASE_DIR, "models")
    OUTPUT_DIR = os.path.join(BASE_DIR, "output")

    TRAIN_CSV = os.path.join(SPLIT_DIR, "train.csv")
    TEST_CSV = os.path.join(SPLIT_DIR, "test.csv")
    VAL_CSV = os.path.join(SPLIT_DIR, "val.csv")
    METADATA = os.path.join(BASE_DIR, "dataset/metadata.csv")

    SAMPLE_RATE = 16000
    BATCH_SIZE = 16
    EPOCHS = 25
    LR = 5e-5

    MODEL_NAME = "openai/whisper-small"

# Main

In [ ]:
import pandas as pd
import os

cfg = Config()

df = pd.read_csv(cfg.METADATA)

df["category"] = df["audio_path"].apply(lambda p: os.path.basename(os.path.dirname(p)))

df["category"] = df["category"].str.lower().str.replace(r"\s+", "", regex=True)

val_categories = {
    "bar_cate_06", "chi_cate_12", "noa_cate_09", "ran_cate_11", "syl_cate_08"
}
test_categories = {
    "bar_cate_14", "chi_cate_19", "noa_cate_18", "ran_cate_05", "syl_cate_11"
}

all_categories = set(df["category"].unique())
missing_val = val_categories - all_categories
missing_test = test_categories - all_categories
if missing_val:
    print("⚠️ WARNING: these validation categories were not found in metadata:", missing_val)
if missing_test:
    print("⚠️ WARNING: these test categories were not found in metadata:", missing_test)

val_df = df[df["category"].isin(val_categories)].reset_index(drop=True)
test_df = df[df["category"].isin(test_categories)].reset_index(drop=True)
train_df = df[~df["category"].isin(val_categories | test_categories)].reset_index(drop=True)

# save
os.makedirs(cfg.SPLIT_DIR, exist_ok=True)

train_df.to_csv(cfg.TRAIN_CSV, index=False)
val_df.to_csv(cfg.VAL_CSV, index=False)
test_df.to_csv(cfg.TEST_CSV, index=False)

print("Category-based Split Done!")
print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

print("\nValidation categories used (per region):")
print(val_df.groupby("region")["category"].unique())

print("\nTest categories used (per region):")
print(test_df.groupby("region")["category"].unique())

In [ ]:
import pandas as pd

train_df = pd.read_csv(cfg.TRAIN_CSV)
val_df = pd.read_csv(cfg.VAL_CSV)
test_df = pd.read_csv(cfg.TEST_CSV)

summary = pd.DataFrame({
    "Train": train_df["region"].value_counts(),
    "Validation": val_df["region"].value_counts(),
    "Test": test_df["region"].value_counts()
}).fillna(0).astype(int)

print(summary)

In [ ]:
import numpy as np

def add_noise(audio, noise_factor=0.005):
    noise = np.random.randn(len(audio))
    return audio + noise_factor * noise

In [ ]:
! pip install librosa

In [ ]:
from torch.utils.data import Dataset
import librosa

class WhisperDataset(Dataset):

    def __init__(self, csv_file, config, processor, augment=False):

        self.df = pd.read_csv(csv_file)

        self.cfg = config
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):

        audio, sr = librosa.load(
            path,
            sr=self.cfg.SAMPLE_RATE
        )

        # remove NaN
        audio = np.nan_to_num(audio)

        # LIMIT LENGTH (VERY IMPORTANT)
        audio = audio[:self.cfg.SAMPLE_RATE * 10]

        return audio

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        path = os.path.join(
            self.cfg.BASE_DIR,
            row["audio_path"]
        )

        # check file
        if not os.path.exists(path):
            raise FileNotFoundError(path)

        audio = self.load_audio(path)

        # FAST AUGMENTATION
        if self.augment:
            if np.random.rand() < 0.6:
                audio = add_noise(audio)

        # feature extraction
        input_features = self.processor.feature_extractor(
            audio,
            sampling_rate=self.cfg.SAMPLE_RATE,
            return_tensors="pt"
        ).input_features[0]

        # labels
        labels = self.processor.tokenizer(
            row["transcript"],
            return_tensors="pt"
        ).input_ids.squeeze()

        return {
            "input_features": input_features,
            "labels": labels,
            "region": row["region"]
        }

In [ ]:
def collate_fn(batch):

    input_features = torch.stack(
        [b["input_features"] for b in batch]
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch],
        batch_first=True,
        padding_value=-100
    )

    regions = [b["region"] for b in batch]

    return {
        "input_features": input_features,
        "labels": labels,
        "region": regions
    }

In [ ]:
!pip install transformers datasets accelerate soundfile

In [ ]:
from transformers import WhisperProcessor
from transformers import WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(cfg.MODEL_NAME)

model = WhisperForConditionalGeneration.from_pretrained(
    cfg.MODEL_NAME
)

model.config.use_cache = False

# force bangla
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="bn",
    task="transcribe"
)

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print("Device:", device)


In [ ]:
from torch.utils.data import DataLoader

train_ds = WhisperDataset(
    cfg.TRAIN_CSV,
    cfg,
    processor,
    augment=False
)

val_ds = WhisperDataset(
    cfg.VAL_CSV,
    cfg,
    processor,
    augment=False
)

test_ds = WhisperDataset(
    cfg.TEST_CSV, 
    cfg, 
    processor, 
    augment=False
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.BATCH_SIZE,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.BATCH_SIZE,
    num_workers=0,
    collate_fn=collate_fn
)


In [ ]:
df = pd.read_csv(cfg.TRAIN_CSV)

missing = []

for path in df["audio_path"]:

    full_path = os.path.join(cfg.BASE_DIR, path)

    if not os.path.exists(full_path):
        missing.append(full_path)

print("Missing Files:", len(missing))

if len(missing) > 0:
    print(missing[:10])

In [ ]:
import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup

train_losses = []
val_losses = []

optimizer = optim.AdamW(
    model.parameters(),
    lr=cfg.LR,
    weight_decay=0.01
)

num_training_steps = len(train_loader) * cfg.EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=500,
    num_training_steps=num_training_steps
)

best_loss = float("inf")
patience = 5  
patience_counter = 0

BEST_MODEL_DIR = os.path.join(cfg.MODEL_DIR, "whisper_best")
LAST_MODEL_DIR = os.path.join(cfg.MODEL_DIR, "whisper_last")

os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(LAST_MODEL_DIR, exist_ok=True)

for epoch in range(cfg.EPOCHS):

    print("🚀 Epoch:", epoch + 1)

    if epoch < 3:
        print("Encoder Frozen for stabilization.")
        for param in model.model.encoder.parameters():
            param.requires_grad = False
    else:
        print("Encoder Unfrozen.")
        for param in model.model.encoder.parameters():
            param.requires_grad = True
            
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader)

    for step, batch in enumerate(progress_bar):
        batch = {
            k: v.to(device)
            for k, v in batch.items()
            if k != "region"
        }

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        progress_bar.set_postfix({"loss": loss.item()})

    avg_train_loss = total_loss / len(train_loader)
    print("Train Loss:", avg_train_loss)

    # VALIDATION
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            batch = {
                k: v.to(device)
                for k, v in batch.items()
                if k != "region"
            }

            outputs = model(**batch)
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print("Val Loss:", avg_val_loss)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        patience_counter = 0
        

        model.save_pretrained(BEST_MODEL_DIR)
        

        if 'processor' in globals():
            processor.save_pretrained(BEST_MODEL_DIR)
        elif 'tokenizer' in globals():
            tokenizer.save_pretrained(BEST_MODEL_DIR)

        print(f"✅ Best Hugging Face format model saved at: {BEST_MODEL_DIR} (Val Loss: {best_loss:.4f})")
    else:
        patience_counter += 1
        print(f"Loss didn't improve. Early stopping counter: {patience_counter}/{patience}")


    model.save_pretrained(LAST_MODEL_DIR)
    if 'processor' in globals():
        processor.save_pretrained(LAST_MODEL_DIR)
    elif 'tokenizer' in globals():
        tokenizer.save_pretrained(LAST_MODEL_DIR)

    if patience_counter >= patience:
        print("Early stopping triggered. Training terminated!")
        break

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8,5))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Val Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss Curve")
plt.legend()
plt.grid()

save_path = os.path.join(cfg.OUTPUT_DIR, "loss_curve.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight') 
print(f"Loss curve graph successfully saved at: {save_path}")

plt.show()

In [ ]:
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8,5))

plt.plot(epochs[:12], train_losses[:12], label="Train Loss")
plt.plot(epochs[:12], val_losses[:12], label="Val Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
! pip install --upgrade pyarrow numpy --force-reinstall

In [ ]:
! pip install evaluate
! pip install jiwer evaluate

In [ ]:
import evaluate
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor


BEST_MODEL_DIR = "models/whisper_best"  
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(BEST_MODEL_DIR)
model = WhisperForConditionalGeneration.from_pretrained(BEST_MODEL_DIR)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="bn", task="transcribe"
)

model.to(device)
model.eval()

print("✅ Whisper model loaded successfully from Hugging Face checkpoint!")

# Metrics Load
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print("✅ Metrics Loaded")

In [ ]:
import re

def clean_bengali_text(text):
    text = re.sub(r'[।,;:!?•\'"()\[\]{}—\-_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
def evaluate_model(loader):
    model.eval()

    preds = []
    refs = []

    with torch.no_grad():
        for batch in loader:
            inputs = batch["input_features"].to(device)

            generated = model.generate(inputs)

            # predictions
            pred_text = processor.tokenizer.batch_decode(
                generated,
                skip_special_tokens=True
            )

            labels = batch["labels"]

            labels = labels.clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id

            ref_text = processor.tokenizer.batch_decode(
                labels,
                skip_special_tokens=True
            )

            cleaned_pred_text = [clean_bengali_text(t) for t in pred_text]
            cleaned_ref_text = [clean_bengali_text(t) for t in ref_text]

            preds.extend(cleaned_pred_text)
            refs.extend(cleaned_ref_text)
            
    wer = wer_metric.compute(predictions=preds, references=refs)
    cer = cer_metric.compute(predictions=preds, references=refs)
    
    word_accuracy = (1 - wer) * 100
    char_accuracy = (1 - cer) * 100

    return wer, cer, word_accuracy, char_accuracy

In [ ]:
import os
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

wer, cer, w_acc, c_acc = evaluate_model(test_loader)

print("FINAL RESULT")
print("FINAL TEST DATA RESULT")
print(f"Word Error Rate (WER)     : {wer:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")
print(f"Word Accuracy (WAcc)      : {w_acc:.2f}%") 
print(f"Character Accuracy (CAcc) : {c_acc:.2f}%") 

txt_save_path = os.path.join(cfg.OUTPUT_DIR, "evaluation_results.txt")

with open(txt_save_path, "w", encoding="utf-8") as f:
    f.write("FINAL RESULT\n")
    f.write("FINAL TEST DATA RESULT\n")
    f.write(f"Word Error Rate (WER)     : {wer:.4f}\n")
    f.write(f"Character Error Rate (CER): {cer:.4f}\n")
    f.write(f"Word Accuracy (WAcc)      : {w_acc:.2f}%\n")
    f.write(f"Character Accuracy (CAcc) : {c_acc:.2f}%\n")

print(f"\nEvaluation results successfully saved as text file at: {txt_save_path}")

In [ ]:
from jiwer import wer as jiwer_wer
from collections import defaultdict
import os

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

region_refs = defaultdict(list)
region_preds = defaultdict(list)

model.eval()

with torch.no_grad():
    for batch in test_loader: 
        region = batch["region"] 
        input_features = batch["input_features"].to(device)

        generated_ids = model.generate(input_features=input_features)

        preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
        labels = batch["labels"]
        labels[labels == -100] = processor.tokenizer.pad_token_id
        refs = processor.batch_decode(labels, skip_special_tokens=True)

        for r, pred, ref in zip(region, preds, refs):
            region_preds[r].append(clean_bengali_text(pred))
            region_refs[r].append(clean_bengali_text(ref))

regional_txt_path = os.path.join(cfg.OUTPUT_DIR, "regional_accuracy_report.txt")

print("\nREGIONAL ACCURACY REPORT")

with open(regional_txt_path, "w", encoding="utf-8") as f:
    f.write("REGIONAL ACCURACY REPORT\n")
    f.write("=========================\n\n")
    
    for region in region_refs:
        region_wer = jiwer_wer(region_refs[region], region_preds[region])
        region_w_acc = max(0, (1 - region_wer) * 100) 
        
        print(f"{region}:")
        print(f"   - WER      : {region_wer:.4f}")
        print(f"   - Accuracy : {region_w_acc:.2f}%")
        
        f.write(f"{region}:\n")
        f.write(f"   - WER      : {region_wer:.4f}\n")
        f.write(f"   - Accuracy : {region_w_acc:.2f}%\n\n")

print(f"\nRegional accuracy report successfully saved as a separate text file at: {regional_txt_path}")

# Word-Level Verification (Val & Test)

This section shows, sentence by sentence, exactly which words the model got right and which it got wrong -- so accuracy isn't just a single WER/CER number, you can *see* the actual matches/mismatches for every prediction.

In [ ]:
import difflib
import pandas as pd

def word_level_diff(ref, pred):
    """Word-by-word alignment between reference and prediction.

    Returns:
        diff_str : human readable string marking matched words with a check
                   mark and mismatches showing ref -> pred
        matched  : number of correctly matched words
        total    : total number of words in the reference
    """
    ref_words = ref.split()
    pred_words = pred.split()

    sm = difflib.SequenceMatcher(None, ref_words, pred_words)

    matched = 0
    parts = []

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            matched += (i2 - i1)
            for w in ref_words[i1:i2]:
                parts.append(f"\u2705{w}")
        elif tag == "replace":
            n = min(i2 - i1, j2 - j1)
            for k in range(n):
                parts.append(f"\u274c{ref_words[i1 + k]}\u2192{pred_words[j1 + k]}")
            if (i2 - i1) > n:
                for w in ref_words[i1 + n:i2]:
                    parts.append(f"\u274c{w}\u2192[missing]")
            if (j2 - j1) > n:
                for w in pred_words[j1 + n:j2]:
                    parts.append(f"\u2795[extra]\u2192{w}")
        elif tag == "delete":
            for w in ref_words[i1:i2]:
                parts.append(f"\u274c{w}\u2192[missing]")
        elif tag == "insert":
            for w in pred_words[j1:j2]:
                parts.append(f"\u2795[extra]\u2192{w}")

    total = len(ref_words)
    return " ".join(parts), matched, total


def verify_predictions(loader, split_name="Validation", print_limit=20, num_beams=5, save_csv=True):
    """Run inference on a loader and show word-level match/mismatch for every sentence.

    print_limit=None -> prints every sample (can be long!)
    num_beams>1       -> beam search decoding, usually a bit more accurate than greedy
    """
    model.eval()

    rows = []
    total_words = 0
    total_matched = 0
    sample_idx = 0

    print("=" * 70)
    print(f"{split_name.upper()} \u2014 WORD-LEVEL VERIFICATION")
    print("=" * 70)

    with torch.no_grad():
        for batch in loader:
            region = batch["region"]
            input_features = batch["input_features"].to(device)

            generated_ids = model.generate(input_features=input_features, num_beams=num_beams)
            preds = processor.batch_decode(generated_ids, skip_special_tokens=True)

            labels = batch["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id
            refs = processor.batch_decode(labels, skip_special_tokens=True)

            for r, pred, ref in zip(region, preds, refs):
                ref_c = clean_bengali_text(ref)
                pred_c = clean_bengali_text(pred)

                diff_str, matched, total = word_level_diff(ref_c, pred_c)
                total_words += total
                total_matched += matched
                sentence_acc = (matched / total * 100) if total > 0 else 0.0

                rows.append({
                    "region": r,
                    "reference": ref_c,
                    "prediction": pred_c,
                    "matched_words": matched,
                    "total_words": total,
                    "word_match_%": round(sentence_acc, 2),
                    "diff": diff_str
                })

                sample_idx += 1
                if print_limit is None or sample_idx <= print_limit:
                    print(f"\n[{sample_idx}] Region: {r}")
                    print(f"  Reference : {ref_c}")
                    print(f"  Prediction: {pred_c}")
                    print(f"  Match     : {diff_str}  ({matched}/{total} words = {sentence_acc:.1f}%)")

    overall_acc = (total_matched / total_words * 100) if total_words > 0 else 0.0
    print("\n" + "-" * 70)
    print(f"{split_name} Overall Word-Level Accuracy: {total_matched}/{total_words} = {overall_acc:.2f}%")
    print("-" * 70)

    result_df = pd.DataFrame(rows)

    if save_csv:
        os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
        csv_path = os.path.join(cfg.OUTPUT_DIR, f"{split_name.lower()}_word_verification.csv")
        result_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"\nFull sentence-by-sentence verification saved to: {csv_path}")

    return result_df

In [ ]:


val_verification_df = verify_predictions(val_loader, split_name="Validation", print_limit=20)
test_verification_df = verify_predictions(test_loader, split_name="Test", print_limit=20)

In [ ]:


import librosa
import numpy as np
import os

audio_path = None

try:
    import google.colab
    from google.colab import files
    print("🎤 your audio file (wav/mp3) upload here:")
    uploaded = files.upload()
    audio_path = list(uploaded.keys())[0]
except ImportError:
    try:
        import ipywidgets as widgets
        from IPython.display import display

        uploader = widgets.FileUpload(accept='audio/*', multiple=False)
        display(uploader)
        print("⬆️ Upload the file and then run this cell again (the path below needs to be corrected if auto-detection fails).")

        if len(uploader.value) > 0:
            file_info = list(uploader.value.values())[0]
            audio_path = "temp_uploaded_audio.wav"
            with open(audio_path, "wb") as f:
                f.write(file_info["content"])
    except Exception:
        pass

    if audio_path is None:
        audio_path = input("Audio file path: ").strip()

print("✅ Using audio file:", audio_path)

SAMPLE_RATE = 16000

audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
audio = np.nan_to_num(audio)
audio = audio[:SAMPLE_RATE * 10]  

input_features = processor.feature_extractor(
    audio,
    sampling_rate=SAMPLE_RATE,
    return_tensors="pt"
).input_features.to(device)

model.eval()
with torch.no_grad():
    generated_ids = model.generate(input_features, num_beams=5)

predicted_text = processor.tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

predicted_text_clean = clean_bengali_text(predicted_text)

print("\n" + "=" * 50)
print("🗣️  PREDICTED TEXT:", predicted_text_clean)
print("=" * 50)

reference_text = input("\n(Optional) সঠিক transcript থাকলে দাও, নাহলে খালি রেখে Enter দাও: ").strip()

if reference_text:
    reference_text_clean = clean_bengali_text(reference_text)
    sample_wer = wer_metric.compute(predictions=[predicted_text_clean], references=[reference_text_clean])
    sample_cer = cer_metric.compute(predictions=[predicted_text_clean], references=[reference_text_clean])

    print(f"\nReference : {reference_text_clean}")
    print(f"Predicted : {predicted_text_clean}")
    print(f"WER       : {sample_wer:.4f}")
    print(f"CER       : {sample_cer:.4f}")

In [ ]:

import os
import re
import torch
import numpy as np
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_DIR = r"D:\Team_34\models\whisper_best"   
SAMPLE_RATE = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(f"❌ Model folder পাওয়া যায়নি: {MODEL_DIR}")

print("⏳ Loading processor & model from:", MODEL_DIR)

processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="bn",
    task="transcribe"
)

model.to(device)
model.eval()
print("✅ Fine-tuned model loaded successfully!")


audio_path = input("Audio file-এর path দাও (e.g. D:\\Team_34\\sentence_10.wav): ").strip().strip('"')

if not os.path.exists(audio_path):
    raise FileNotFoundError(f"❌ File not found: {audio_path}")

print("✅ Using audio file:", audio_path)

# ---- 3) Audio load + feature extraction ----
audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
audio = np.nan_to_num(audio)
audio = audio[:SAMPLE_RATE * 10]  # max 10 sec

input_features = processor.feature_extractor(
    audio,
    sampling_rate=SAMPLE_RATE,
    return_tensors="pt"
).input_features.to(device)

# ---- 4) Predict ----
with torch.no_grad():
    generated_ids = model.generate(input_features, num_beams=5)

predicted_text = processor.tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

def clean_bengali_text(text):
    text = re.sub(r'[।,;:!?•\'"()\[\]{}—\-_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

predicted_text_clean = clean_bengali_text(predicted_text)

print("\n" + "=" * 50)
print("🗣️  PREDICTED TEXT:", predicted_text_clean)
print("=" * 50)


reference_text = input("\n(Optional) সঠিক transcript থাকলে দাও, নাহলে খালি রেখে Enter দাও: ").strip()

if reference_text:
    import evaluate
    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")

    reference_text_clean = clean_bengali_text(reference_text)
    sample_wer = wer_metric.compute(predictions=[predicted_text_clean], references=[reference_text_clean])
    sample_cer = cer_metric.compute(predictions=[predicted_text_clean], references=[reference_text_clean])

    print(f"\nReference : {reference_text_clean}")
    print(f"Predicted : {predicted_text_clean}")
    print(f"WER       : {sample_wer:.4f}")
    print(f"CER       : {sample_cer:.4f}")